# Optional: backfill the past week of weather

**DSC 592 · Lecture 19 (optional)**

Your hourly job only collects weather **from the moment it starts**. A **backfill** loads data for a period **in the past**: here, the last 7 days of hourly weather.

The trick: write the past hours into the landing volume in **exactly the same layout and shape** as `fetch_weather.ipynb` does. Then your existing pipeline loads them with no code change: Auto Loader just sees new files.

```text
landing/weather/yyyy/mm/dd/hh/<city>.json      same folders, same file names, same JSON shape
```

## 1. Settings

In [ ]:
import os, json
from datetime import datetime, timezone
import pandas as pd
import requests

LANDING   = "/Volumes/dsc592/weather/landing/weather"
PAST_DAYS = 7                     # how far back (the forecast API allows up to 92 days)

# Replace with your team's cities, for example:
# cities = pd.read_csv("/Workspace/Users/<you>/weather-teamN/ingest/cities.csv")
cities = pd.DataFrame([{"city": "Wilmington", "lat": 34.2257, "lon": -77.9447}])
display(cities)

## 2. One request per city: past hours as arrays

`past_days` asks the API for the hours **before** today as well. The answer has the same `hourly` block as in Lecture 16: parallel arrays, where `time[i]` goes with `temperature_2m[i]`.

In [ ]:
URL  = "https://api.open-meteo.com/v1/forecast"
VARS = ["temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m", "weather_code"]

def fetch_past(lat, lon):
    r = requests.get(URL, timeout=30, params={
        "latitude": lat, "longitude": lon, "hourly": ",".join(VARS),
        "past_days": PAST_DAYS, "forecast_days": 1, "timezone": "UTC"})
    r.raise_for_status()
    return r.json()

sample = fetch_past(cities.lat[0], cities.lon[0])
print(len(sample["hourly"]["time"]), "hours")      # (PAST_DAYS + 1) x 24
print(sample["hourly"]["time"][:3])

## 3. Write one file per city per past hour

Each past hour becomes one landing file, shaped like a live one: that hour's values go into `payload.current`, so `silver.sql` reads them unchanged.

Two rules make the backfill safe to rerun:

- **Only past hours:** stop at the current hour (the rest of the day is a forecast)
- **Never overwrite:** if a file already exists (your live job wrote it), skip it

In [ ]:
now      = datetime.now(timezone.utc)
this_hour = now.replace(minute=0, second=0, microsecond=0)
written = skipped = 0

for row in cities.itertuples():
    data   = fetch_past(row.lat, row.lon)
    hourly = data["hourly"]
    slug   = row.city.lower().replace(" ", "_")

    for i, t in enumerate(hourly["time"]):
        ts = datetime.fromisoformat(t).replace(tzinfo=timezone.utc)
        if ts >= this_hour:
            break                                            # only hours that are over

        folder = f"{LANDING}/{ts:%Y/%m/%d/%H}"
        path   = f"{folder}/{slug}.json"
        if os.path.exists(path):
            skipped += 1                                     # keep the live file
            continue

        current = {"time": t, "interval": 3600}
        for v in VARS:
            current[v] = hourly[v][i]
        record = {"city": row.city, "source": "open-meteo backfill",
                  "run_hour": ts.isoformat(),                # the hour the data is about
                  "fetched_at": now.isoformat(timespec="seconds"),
                  "payload": {"latitude": data["latitude"], "longitude": data["longitude"],
                              "timezone": data["timezone"], "current": current}}

        os.makedirs(folder, exist_ok=True)
        with open(path, "w") as f:
            json.dump(record, f)
        written += 1

print(written, "files written,", skipped, "skipped (already there)")

## 4. Load it: run your pipeline

Run `weather-ETL` (or **Run now** on `weather-hourly`). Auto Loader loads only the new files; silver and gold update as usual.

Then check how far back silver now goes:

In [ ]:
%sql
SELECT city, MIN(obs_ts) AS first_obs, MAX(obs_ts) AS last_obs, COUNT(*) AS n_obs
FROM dsc592.weather.silver_observations
GROUP BY city
ORDER BY city

## Notes

- **Why the folder uses the data's hour, not today:** a backfill describes the past, so each file goes where a live pull for that hour would have put it. Live ingest and backfill then share one layout.
- **Rerunning is safe:** existing files are skipped, and Auto Loader never loads a file twice.
- **Longer history:** the forecast API's `past_days` goes back about 3 months. For older data, Open-Meteo has a separate **archive** API (`archive-api.open-meteo.com`); the weather-pipeline repo's `ingest/backfill_weather_daily.ipynb` uses it for daily values.
- **For your project:** check whether your source has a history endpoint. It is the easiest way to get enough data for a dashboard trend or a model (Option B).